# verifying classes and functions

CoHDL code is usually structured using Python functions and classes. To verify these constructs we can wrap them in Entities as we have done in all examples so far. This works but can be inconvenient because a layer of indirection is added on top of the tested functionality.

Another approach is to place the tested code directly in the YosysTestCase architecture.

In [1]:
# basic setup of jupyter notebook

from __future__ import annotations

from example_util.jupyter_util import display_vcd

import cohdl

# When an exception occurs during compilation,
# cohdl inserts fake stack frames into the exception traceback.
# This does not work properly inside jupyter notebooks.
cohdl.use_pretty_traceback(False)

As an example we will verify the following `Incrementer` class. It has an internal variable that is incremented whenever `increment` is called. The `value` method obtains the current counter state. The counter is cleared when `reset` is called.

In [2]:
from cohdl import Entity, Bit, Port, Unsigned, Signal
from cohdl import std

class Incrementer:
    def __init__(self):
        self._cnt = Signal[Unsigned[8]](0)

    def value(self):
        return self._cnt

    def increment(self, val=1):
        self._cnt <<= self._cnt + val

    def reset(self):
        self._cnt <<= 0

To verify the class we need a way to translate the action of calling Python methods to a representation Yosys can understand. The easiest way to do that is to place every method in an if-statement guarded with an `Anyseq` signal. That way a true value corresponds to a performed call.

We can then use this value in assertions and assumptions to check the output behavior and restrict when calls are allowed.

In [3]:
from cohdl_yosys import YosysTestCase, YosysParams
from cohdl_yosys.formal import (
    set_default_ctx,
    Anyseq,
    always,
    When,
    prev,
)


# Yosys and by extension the YosysTestCase class requires
# an entity to test. We just provide an empty dummy
# because all tested logic is in the architecture method.
class Dummy(Entity):
    clk = Port.input(Bit)


class Check_Incrementer(YosysTestCase, entity=Dummy):
    _yosys_params_ = YosysParams(bmc=True, quiet=True, clean_build_dir=True)

    def architecture(self, dut: Dummy):
        ctx = set_default_ctx(clk=std.Clock(dut.clk))

        inc = Incrementer()

        inp_val = Anyseq[Unsigned[8]]()
        called_inc = Anyseq[bool]()
        called_reset = Anyseq[bool]()

        @ctx
        def proc_use_incrementer():
            # use Anyseq signals to provide input
            # and control method calls

            if called_inc:
                inc.increment(inp_val)

            if called_reset:
                inc.reset()

        @std.concurrent
        def formal_properties():

            always["reset_clears_value"](When(called_reset).then_next(inc.value() == 0))

            val = inc.value()

            always["increment_works"](
                When(called_inc).then_next(val == prev(val + inp_val)),
                sync_abort=called_reset,
            )


if Check_Incrementer().test_formal_properties(return_on_error=True):
    print("formal check has passed")

formal check has passed


Declaring an `Anyseq[bool]` for every method and wrapping it in an if-statement is a common, repetitive task. cohdl_yosys provides the `CtxWrapper` class with the `maybe_call` method to do it for us.

The first argument of `maybe_call` is a callable object. It will be invoked with the remaining arguments. The return value is the `Anyseq` signal that controls the call.

In [4]:
from cohdl_yosys.formal import CtxWrapper

class Check_Incrementer(YosysTestCase, entity=Dummy):
    _yosys_params_ = YosysParams(bmc=True, cover=True, quiet=True, clean_build_dir=True)

    def architecture(self, dut: Dummy):
        ctx = CtxWrapper(set_default_ctx(clk=std.Clock(dut.clk)))

        inc = Incrementer()

        inp_val = Anyseq[Unsigned[8]]()
        called_inc = ctx.maybe_call(inc.increment, inp_val)
        called_reset = ctx.maybe_call(inc.reset)

        @std.concurrent
        def formal_properties():

            always["reset_clears_value"](When(called_reset).then_next(inc.value() == 0))

            val = inc.value()

            always["increment_works"](
                When(called_inc).then_next(val == prev(val + inp_val)),
                sync_abort=called_reset,
            )

if Check_Incrementer().test_formal_properties(return_on_error=True):
    print("formal check has passed")

formal check has passed
